# Article 1 — Baseline viability pilot

This notebook executes the pre-registered diagnostic path only: B0 retrieval overlap, B1 lightweight features, B3 exact same-source deduplication, and B4 canonical order. It does not run the proposed SBEV method or an NLI/LLM. All harmful labels generated here are weak labels awaiting audit.

In [ ]:
from pathlib import Path
import json, subprocess, sys
# Each Colab notebook can receive its own runtime. Clone if this runtime has no source tree.
REPO_URL = 'https://github.com/ByvatovS/rag-evidence-robustness.git'
SOURCE_DIR = Path('/content/rag-evidence-robustness')
if not SOURCE_DIR.is_dir():
    subprocess.run(['git', 'clone', '--branch', 'main', REPO_URL, str(SOURCE_DIR)], check=True)
%cd /content/rag-evidence-robustness
WORKDIR = '/content/article1_pilot'
COPY_TO_DRIVE = False
DRIVE_ROOT = Path('/content/drive/MyDrive/llm-rag-research/article1')
!{sys.executable} -m pip install -e '.[pilot]'


In [ ]:
# The order matters: originals -> variants -> train/dev fitting -> one held-out evaluation.
!{sys.executable} -m evidence_robustness.cli prepare-data --config configs/pilot.yaml --workdir {WORKDIR}
!{sys.executable} -m evidence_robustness.cli build-perturbations --config configs/pilot.yaml --workdir {WORKDIR}
!{sys.executable} -m evidence_robustness.cli fit-baselines --config configs/pilot.yaml --workdir {WORKDIR}
!{sys.executable} -m evidence_robustness.cli evaluate --config configs/pilot.yaml --workdir {WORKDIR}


In [ ]:
summary_path = Path(WORKDIR) / 'reports' / 'pilot_baseline_summary.json'
summary = json.loads(summary_path.read_text(encoding='utf-8'))
print(json.dumps(summary, ensure_ascii=False, indent=2, sort_keys=True))
if COPY_TO_DRIVE:
    import shutil
    target = DRIVE_ROOT / 'runs' / 'baseline_pilot'
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(WORKDIR, target, dirs_exist_ok=True)
    print('Copied artefacts to', target)


## Send these artifacts back

Send `processed/pilot_manifest.json`, `perturbations/perturbation_manifest.json`, `models/diagnostic_baselines_manifest.json`, and `reports/pilot_baseline_summary.json`, plus any command error. Do **not** interpret the reported weak-label scores as a scientific result before the manual audit specified in `docs/annotation_guide.md`.